In [ ]:
"""
Power Grid Oscillation Reconstruction Pipeline
================================================

Adapted SENDAI framework for power grid PMU data:
  - LF pathway: SINDy-SHRED (GRU encoder -> latent -> SINDy regularization -> decoder)
                 or vanilla SHRED (LSTM encoder -> latent -> decoder)
  - HF pathway: Hierarchical temporal-frequency peeling layers (1D Conv + MLP)

Maps p observed PMU sensors -> M total PMU locations.

Usage:
    python main.py [--scenario iberian_event] [--lf-mode sindy_shred]
"""
# ----------------------------
# Blocchi per evitare crash Fortran e gestire Ctrl+C
# ----------------------------
import os
import signal

# Limita i thread MKL / OpenMP
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"

# Gestione Ctrl+C
def handle_sigint(sig, frame):
    print("\nClosed.")
    exit(0)

signal.signal(signal.SIGINT, handle_sigint)

import json
import math
import warnings
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Optional, Tuple, Dict
from collections import OrderedDict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import ConcatDataset, DataLoader, Dataset, TensorDataset
from sklearn.preprocessing import MinMaxScaler
from scipy.signal import butter, filtfilt
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import pandas as pd

import sys

warnings.filterwarnings('ignore')


class TeeLogger:
    """Duplicate stdout to both console and a log file."""
    def __init__(self, filepath):
        self.terminal = sys.stdout
        self.log = open(filepath, "w")
    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)
    def flush(self):
        self.terminal.flush()
        self.log.flush()
    def close(self):
        self.log.close()
        sys.stdout = self.terminal

###############################################################################
# CONFIGURATION
###############################################################################

@dataclass
class PipelineConfig:
    """Full configuration for the reconstruction pipeline."""
    # Data
    data_dir: str = "data/synthetic/iberian_event"
    scenario: str = "iberian_event"
    max_samples: Optional[int] = 5000   # Truncate data to first N samples after data_start_idx (None = use all)
    data_start_idx: int = 2100#2800  #20050         # Starting row index in CSV for reconstruction data
    data_end_idx: Optional[int] = 3000#3700 # Ending row index in CSV for reconstruction data (None = data_start_idx + max_samples)
    inference_extra_idx: Optional[int] = 3600#4000 # Ending row index for inference-only data beyond data_end_idx (None = no inference)

    # ---------- LF pathway mode ----------
    # "sindy_shred" = authentic SINDy-SHRED (GRU + SINDy regularization)
    # "vanilla_shred" = plain SHRED (LSTM encoder + MLP decoder, no SINDy)
    # Note: please change in parser.add_argument for LF mode switch, not here
    lf_mode: str = "sindy_shred"

    # LF pathway (shared)
    latent_dim: int = 8
    lags: int = 400
    lstm_hidden: int = 64          # also used as GRU hidden size
    lstm_layers: int = 2
    decoder_hidden: List[int] = field(default_factory=lambda: [256, 256])
    dropout: float = 0.1
    lf_epochs: int = 500
    lf_batch_size: int = 128
    lf_lr: float = 1e-4
    lf_patience: int = 30

    # SINDy-SHRED specific
    sindy_poly_order: int = 1
    sindy_include_sine: bool = False # False
    sindy_regularization: float = 5 # 10.0
    sindy_threshold: float = 0#0.025 #0.25
    sindy_thres_epoch: int = 10 #20
    sindy_warmup_epochs: int = 20 #30

    # Post-hoc SINDy identification threshold
    sindy_posthoc_threshold: float = 0#0.02 #0.1

    # LF target smoothing: low-pass filter cutoff for LF training target.
    # The LF model is trained on smoothed data so latent space captures only
    # slow dynamics (amenable to SINDy). HF pathway uses full (unsmoothed) data.
    lf_cutoff_hz: float = 0.08  # Hz; set below lowest interarea mode (~0.15 Hz)

    # HF pathway
    t_hf: int = 200
    hf_conv_channels: List[int] = field(default_factory=lambda: [32, 64])
    hf_conv_kernel: int = 5
    hf_mlp_hidden: List[int] = field(default_factory=lambda: [128, 128])
    n_peel_layers: int = 10
    hf_epochs: int = 400
    hf_batch_size: int = 128
    hf_lr: float = 1e-4
    hf_patience: int = 30
    hf_warmup_epochs: int = 80
    lambda_sparse: float = 0.1
    lambda_sparse_finetune: float = 0.002
    finetune_epochs: int = 150
    max_freq_hz: float = 5.0
    lambda_mag: float = 0.001
    mag_threshold: float = 0.01

    # General
    seed: int = 42
    device: str = "auto"
    train_fraction: float = 0.7
    valid_fraction: float = 0.15

    # Set dynamically after loading data
    fs: float = 10.0
    n_pmu_total: int = 22
    n_pmu_observed: int = 10

    def resolve_device(self):
        if self.device == "auto":
            return torch.device("cuda" if torch.cuda.is_available() else "cpu")
        return torch.device(self.device)


###############################################################################
# DATASETS
###############################################################################

class PMUTimeSeriesDataset(Dataset):
    """LF pathway dataset. Input: [lags, p], Output: [M]."""
    def __init__(self, sensor_sequences, full_states):
        self.X = torch.tensor(sensor_sequences, dtype=torch.float32)
        self.Y = torch.tensor(full_states, dtype=torch.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.Y[idx]


class HFWindowDataset(Dataset):
    """HF pathway dataset. Input: [T_hf, p], Output: [T_hf, M]."""
    def __init__(self, sensor_windows, full_windows):
        self.X = torch.tensor(sensor_windows, dtype=torch.float32)
        self.Y = torch.tensor(full_windows, dtype=torch.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.Y[idx]


###############################################################################
# SINDY LIBRARY (for SINDy-SHRED regularization during training)
###############################################################################

def sindy_library(z, poly_order, include_sine=False):
    """Build the SINDy feature library Theta(z).

    Args:
        z: [B, latent_dim] tensor of latent states
        poly_order: max polynomial order
        include_sine: whether to append sin(z) columns

    Returns:
        Theta: [B, library_dim] feature matrix
    """
    B, n = z.shape
    columns = [torch.ones(B, 1, device=z.device)]  # constant term

    # Order 1
    if poly_order >= 1:
        columns.append(z)

    # Order 2
    if poly_order >= 2:
        for i in range(n):
            for j in range(i, n):
                columns.append((z[:, i] * z[:, j]).unsqueeze(1))

    # Order 3
    if poly_order >= 3:
        for i in range(n):
            for j in range(i, n):
                for k in range(j, n):
                    columns.append((z[:, i] * z[:, j] * z[:, k]).unsqueeze(1))

    if include_sine:
        columns.append(torch.sin(z))

    return torch.cat(columns, dim=1)


def sindy_library_size(latent_dim, poly_order, include_sine=False):
    """Compute the number of columns in the SINDy library."""
    n = latent_dim
    size = 1  # constant
    if poly_order >= 1:
        size += n
    if poly_order >= 2:
        size += n * (n + 1) // 2
    if poly_order >= 3:
        size += n * (n + 1) * (n + 2) // 6
    if include_sine:
        size += n
    return size


###############################################################################
# LF PATHWAY: SHRED Network (supports GRU or LSTM)
###############################################################################

class SHREDNet(nn.Module):
    """
    Recurrent encoder -> latent -> MLP decoder -> full state reconstruction.

    When mode='sindy_shred', uses GRU and exposes SINDy coefficient matrix.
    When mode='vanilla_shred', uses LSTM (original behaviour).
    """
    def __init__(self, n_sensors, state_dim, latent_dim,
                 hidden_size=64, n_layers=2,
                 decoder_hidden=(256, 256), dropout=0.1,
                 mode="sindy_shred",
                 sindy_poly_order=2, sindy_include_sine=True): # False
        super().__init__()
        self.n_sensors = n_sensors
        self.state_dim = state_dim
        self.latent_dim = latent_dim
        self.mode = mode

        # ---------- Encoder ----------
        if mode == "sindy_shred":
            self.rnn = nn.GRU(n_sensors, hidden_size, n_layers,
                              batch_first=True,
                              dropout=dropout if n_layers > 1 else 0)
        else:
            self.rnn = nn.LSTM(n_sensors, hidden_size, n_layers,
                               batch_first=True,
                               dropout=dropout if n_layers > 1 else 0)
        self.rnn_to_latent = nn.Linear(hidden_size, latent_dim)

        # ---------- Decoder ----------
        layers = []
        in_dim = latent_dim
        for h in decoder_hidden:
            layers.extend([nn.Linear(in_dim, h), nn.ReLU(), nn.Dropout(dropout)])
            in_dim = h
        layers.append(nn.Linear(in_dim, state_dim))
        self.decoder = nn.Sequential(*layers)

        # ---------- SINDy coefficient matrix (only for sindy_shred) ----------
        if mode == "sindy_shred":
            lib_dim = sindy_library_size(latent_dim, sindy_poly_order, sindy_include_sine)
            # Xi: [library_dim, latent_dim]
            self.sindy_coefficients = nn.Parameter(
                torch.randn(lib_dim, latent_dim) * 0.01
            )
            self.sindy_poly_order = sindy_poly_order
            self.sindy_include_sine = sindy_include_sine
            # Mask for thresholding (1 = active, 0 = pruned)
            self.register_buffer('sindy_mask', torch.ones(lib_dim, latent_dim))

    def encode(self, x):
        """Encode sensor sequence -> latent vector (last time step)."""
        out, _ = self.rnn(x)
        return self.rnn_to_latent(out[:, -1, :])

    def encode_sequence(self, x):
        """Encode sensor sequence -> latent trajectory at every time step.
        Returns: [B, T, latent_dim]
        """
        out, _ = self.rnn(x)
        return self.rnn_to_latent(out)

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        return self.decode(self.encode(x))

    def get_latent(self, x):
        self.eval()
        with torch.no_grad():
            return self.encode(x).cpu().numpy()

    # ---------- SINDy helpers ----------

    def sindy_predict_dz(self, z):
        """Predict dz/dt = Theta(z) @ (Xi * mask)."""
        Theta = sindy_library(z, self.sindy_poly_order, self.sindy_include_sine)
        Xi_masked = self.sindy_coefficients * self.sindy_mask
        return Theta @ Xi_masked

    def apply_sindy_threshold(self, threshold):
        """STLS step: threshold small coefficients, then refit survivors.

        Implements the Sequential Thresholded Least Squares algorithm
        (Brunton et al., 2016 / Kutz SINDy architecture):
          1. Identify coefficients with |Xi_ij| < threshold and mask them out.
          2. Zero the pruned coefficient values so they carry no state.
          3. Call stls_refit() if training data (Theta, dz_dt) is cached.
        """
        with torch.no_grad():
            new_mask = (self.sindy_coefficients.abs() >= threshold).float()
            self.sindy_mask.copy_(new_mask)
            # Zero out pruned coefficients so they don't retain stale values
            self.sindy_coefficients.data.mul_(new_mask)

    def stls_refit(self, z_all, dz_dt_all):
        """Least-squares refit of active SINDy coefficients on training data.

        Given the full training latent states z_all and their time derivatives
        dz_dt_all, re-solve for each latent dimension j independently:
            Xi_active_j = argmin || Theta[:, active_j] @ xi_j - dz_dt_j ||^2

        This is the key second half of each STLS iteration: after pruning,
        the surviving coefficients must be re-estimated so they absorb the
        variance previously shared with the pruned terms.

        Args:
            z_all:    [N, latent_dim]  latent states (detached tensor on device)
            dz_dt_all:[N, latent_dim]  time derivatives (detached tensor on device)
        """
        with torch.no_grad():
            Theta = sindy_library(z_all, self.sindy_poly_order,
                                  self.sindy_include_sine)  # [N, lib_dim]
            lib_dim, latent_dim = self.sindy_coefficients.shape

            for j in range(latent_dim):
                active_j = self.sindy_mask[:, j].bool()          # [lib_dim]
                n_active = active_j.sum().item()
                if n_active == 0:
                    continue

                Theta_active = Theta[:, active_j]                # [N, n_active]
                target_j = dz_dt_all[:, j]                       # [N]

                # Solve normal equations: xi* = (Theta^T Theta)^{-1} Theta^T target
                # lstsq is more numerically stable
                sol = torch.linalg.lstsq(Theta_active, target_j.unsqueeze(1))
                xi_fit = sol.solution.squeeze(1)                 # [n_active]

                # Write back into the full coefficient matrix
                self.sindy_coefficients.data[:, j].zero_()
                self.sindy_coefficients.data[active_j, j] = xi_fit


###############################################################################
# HF PATHWAY: Temporal Peeling Layer
###############################################################################

class HFPeelLayer(nn.Module):
    """Single HF layer: 1D Conv temporal encoder + per-timestep MLP decoder."""
    def __init__(self, n_observed, n_total, t_hf,
                 conv_channels=(32, 64), conv_kernel=5,
                 mlp_hidden=(128, 128), dropout=0.1, init_scale=0.1):
        super().__init__()
        conv_layers = []
        in_ch = n_observed
        for out_ch in conv_channels:
            pad = (conv_kernel - 1) // 2
            conv_layers.extend([
                nn.Conv1d(in_ch, out_ch, conv_kernel, padding=pad),
                nn.ReLU(), nn.Dropout(dropout)])
            in_ch = out_ch
        self.conv = nn.Sequential(*conv_layers)
        self.d_hidden = conv_channels[-1]

        mlp = []
        in_dim = self.d_hidden
        for h in mlp_hidden:
            mlp.extend([nn.Linear(in_dim, h), nn.ReLU(), nn.Dropout(dropout)])
            in_dim = h
        mlp.append(nn.Linear(in_dim, n_total))
        self.mlp = nn.Sequential(*mlp)

        self.gamma = nn.Parameter(torch.tensor(init_scale))

    def forward(self, x):
        h = self.conv(x.transpose(1, 2)).transpose(1, 2)
        B, T, D = h.shape
        out = self.mlp(h.reshape(B*T, D)).reshape(B, T, -1)
        return self.gamma * out


###############################################################################
# FULL MODEL
###############################################################################

class FullReconstructionModel(nn.Module):
    """LF + HF combined pipeline."""
    def __init__(self, lf_model, sensor_indices, n_total, t_hf,
                 n_peel_layers=2, hf_conv_channels=(32, 64),
                 hf_conv_kernel=5, hf_mlp_hidden=(128, 128), dropout=0.1):
        super().__init__()
        self.lf_model = lf_model
        self.register_buffer('sensor_indices',
                             torch.tensor(sensor_indices, dtype=torch.long))
        self.n_total = n_total
        self.n_observed = len(sensor_indices)
        self.t_hf = t_hf

        self.hf_layers = nn.ModuleList([
            HFPeelLayer(self.n_observed, n_total, t_hf,
                        conv_channels=list(hf_conv_channels),
                        conv_kernel=hf_conv_kernel,
                        mlp_hidden=list(hf_mlp_hidden),
                        dropout=dropout,
                        init_scale=0.1 / (i + 1))
            for i in range(n_peel_layers)
        ])

    def forward_lf(self, sensor_history):
        return self.lf_model(sensor_history)

    def forward_hf(self, obs_residual_window, n_layers=None):
        """Apply HF layers sequentially to residual windows."""
        n = n_layers or len(self.hf_layers)
        corrections = []
        residual = obs_residual_window
        for i in range(n):
            c = self.hf_layers[i](residual)
            corrections.append(c)
            residual = obs_residual_window - sum(corrections)[:, :, self.sensor_indices]
        return sum(corrections), corrections


###############################################################################
# LOSS FUNCTIONS
###############################################################################

def temporal_freq_sparsity(signal, fs, max_freq_hz):
    """L1/L2 sparsity on in-band temporal FFT + out-of-band penalty."""
    fft = torch.fft.rfft(signal, dim=1)
    mag = torch.abs(fft)
    freqs = torch.fft.rfftfreq(signal.shape[1], d=1.0/fs, device=signal.device)

    in_band = (freqs <= max_freq_hz).float()
    out_band = 1.0 - in_band

    in_mag = mag * in_band[None, :, None]
    in_flat = in_mag.reshape(signal.shape[0], -1)
    l1 = in_flat.sum(-1)
    l2 = (in_flat**2).sum(-1).sqrt().clamp(min=1e-8)
    sparsity = (l1 / l2).mean()

    out_mag = mag * out_band[None, :, None]
    out_e = (out_mag.reshape(signal.shape[0], -1)**2).sum(-1)
    tot_e = (mag.reshape(signal.shape[0], -1)**2).sum(-1).clamp(min=1e-8)
    oob_penalty = (out_e / tot_e).mean()

    return sparsity + 100.0 * oob_penalty


def mag_penalty(correction, threshold):
    return F.relu(correction.abs().mean() - threshold) ** 2


###############################################################################
# DATA PREPARATION
###############################################################################

class DataPreparer:
    """Load synthetic data and prepare all datasets."""

    def __init__(self, config: PipelineConfig):
        self.cfg = config
        self.scaler = None
        self.metadata = None

    def load_and_prepare(self):
        data_dir = Path(self.cfg.data_dir)

        # Load CSV with header row: PMU0, PMU1, ...
        df = pd.read_csv(data_dir / "output_unico_spagna_2016.csv")

        # Extract full numeric matrix (keep for inference truth later)
        x_csv_full = df.values.astype(np.float32)
        #x_csv_full = df.iloc[2499:4000,:].values.astype(np.float32)

        # Apply data windowing: select [data_start_idx : data_end_idx] for reconstruction
        start_idx = self.cfg.data_start_idx
        end_idx = self.cfg.data_end_idx
        if end_idx is None:
            if self.cfg.max_samples is not None:
                end_idx = start_idx + self.cfg.max_samples
            else:
                end_idx = len(x_csv_full)
        x_full = x_csv_full[start_idx:end_idx]
        print(f"Data window: CSV rows [{start_idx}:{end_idx}] "
              f"({end_idx - start_idx} samples)")

        # Store inference truth (beyond reconstruction window) for forward prediction plot
        infer_end = self.cfg.inference_extra_idx
        if infer_end is not None and infer_end > end_idx:
            self.inference_truth_raw = x_csv_full[end_idx:infer_end]
            print(f"Inference window: CSV rows [{end_idx}:{infer_end}] "
                  f"({infer_end - end_idx} samples)")
        else:
            self.inference_truth_raw = None

        # All PMUs are present in the file
        M = x_full.shape[1]

        # If ALL PMUs are observed:
        #sensor_locs = np.arange(M)
        M = x_full.shape[1]
        sensor_locs = np.linspace(0, M - 1, 10, dtype=int)

        # Or if only some PMUs are observed, define them manually:
        # sensor_locs = np.array([0, 3, 5, ...])  # example

        # If metadata.json still exists, keep this:
        with open(data_dir / "metadata.json") as f:
            self.metadata = json.load(f)

        raw = np.load(data_dir / "raw_data.npz")
        with open(data_dir / "metadata.json") as f:
            self.metadata = json.load(f)

        #x_full = raw["x_to_fit"]
        #sensor_locs = raw["sensor_locations"]

        T, M = x_full.shape
        p = len(sensor_locs)
        dt = self.metadata["sampling_interval_s"]
        fs = self.metadata["fs_hz"]

        # Optional: truncate to first max_samples time steps
        if self.cfg.max_samples is not None and self.cfg.max_samples < T:
            x_full = x_full[:self.cfg.max_samples]
            T = self.cfg.max_samples
            print(f"Truncated to max_samples={T}")

        self.cfg.fs = fs
        self.cfg.n_pmu_total = M
        self.cfg.n_pmu_observed = p
        self.cfg.max_freq_hz = min(self.cfg.max_freq_hz, fs / 2.0)

        print(f"Loaded: T={T}, M={M}, p={p}, dt={dt}s, fs={fs}Hz")

        #self.gt_lf = raw["lf_component"]
        #self.gt_hf = raw["hf_component"]
        self.gt_latent = raw["latent_trajectory"]
        if self.cfg.max_samples is not None and self.gt_latent is not None:
            self.gt_latent = self.gt_latent[:self.cfg.max_samples]
        #self.gt_latent = None

        # --- Low-pass filter for LF training target ---
        # The LF model trains on smoothed data so its latent space captures
        # only slow dynamics (clean for SINDy). HF pathway uses full signal.

        # Smoothing on LF data, x_smooth is passed onto the subsequence LF model pipeline
        cutoff = self.cfg.lf_cutoff_hz
        #nyq = fs / 2.0
        nyq = 10.0
        if cutoff < nyq * 0.95:
            b, a = butter(4, cutoff / nyq, btype='low')
            x_smooth = np.zeros_like(x_full)
            for col in range(M):
                x_smooth[:, col] = filtfilt(b, a, x_full[:, col])
            print(f"LF target: low-pass filtered at {cutoff} Hz "
                  f"(Nyquist={nyq} Hz)")
        else:
            x_smooth = x_full.copy()
            print("LF target: no filtering (cutoff >= Nyquist)")

        lags = self.cfg.lags
        n_usable = T - lags
        n_train = int(n_usable * self.cfg.train_fraction)
        n_valid = int(n_usable * self.cfg.valid_fraction)

        # Scaler fit on smoothed training data (LF pathway)
        self.scaler = MinMaxScaler()
        self.scaler.fit(x_smooth[:n_train + lags])
        x_smooth_scaled = self.scaler.transform(x_smooth)

        # Also scale the full (unsmoothed) signal with the SAME scaler
        # so LF predictions and full signal are in the same scale space
        x_full_scaled = self.scaler.transform(x_full)

        # LF datasets: input = observed sensors from FULL signal (realistic),
        #              target = smoothed full state (clean for SINDy)
        all_in = np.zeros((n_usable, lags, p))
        all_out_lf = np.zeros((n_usable, M))
        for i in range(n_usable):
            all_in[i] = x_full_scaled[i:i+lags, sensor_locs]
            all_out_lf[i] = x_smooth_scaled[i+lags-1]

        self.lf_train = PMUTimeSeriesDataset(all_in[:n_train], all_out_lf[:n_train])
        self.lf_valid = PMUTimeSeriesDataset(all_in[n_train:n_train+n_valid],
                                              all_out_lf[n_train:n_train+n_valid])
        self.lf_test = PMUTimeSeriesDataset(all_in[n_train+n_valid:],
                                             all_out_lf[n_train+n_valid:])

        self.x_scaled = x_full_scaled      # full signal (for HF pathway)
        self.x_smooth_scaled = x_smooth_scaled  # smoothed (LF targets)
        self.x_full = x_full
        self.sensor_locs = sensor_locs
        self.dt = dt
        self.fs = fs
        self.n_train = n_train
        self.n_valid = n_valid
        self.n_test = n_usable - n_train - n_valid

        print(f"Splits: train={n_train}, valid={n_valid}, test={self.n_test}")
        return self

    def prepare_hf_windows(self, lf_model, device):
        """Compute LF predictions and create HF window datasets.

        HF target = full_signal_scaled - lf_prediction
        (full signal includes HF + noise; LF prediction is smooth)
        """
        cfg = self.cfg
        t_hf = cfg.t_hf
        lags = cfg.lags

        lf_model.eval()
        all_X = torch.cat([self.lf_train.X, self.lf_valid.X, self.lf_test.X])
        preds = []
        for i in range(0, len(all_X), 256):
            batch = all_X[i:i+256].to(device)
            with torch.no_grad():
                preds.append(lf_model(batch).cpu().numpy())
        lf_preds = np.vstack(preds)

        # Use full (unsmoothed) scaled signal as the ground truth for HF residual
        # x_scaled is the full signal; lf_preds approximate the smoothed target
        N = len(lf_preds)
        n_win = N - t_hf + 1
        sensor_locs = self.sensor_locs

        obs_w = np.zeros((n_win, t_hf, len(sensor_locs)))
        full_w = np.zeros((n_win, t_hf, cfg.n_pmu_total))

        for i in range(n_win):
            # Full signal at these time steps (unsmoothed, scaled)
            full_slice = self.x_scaled[lags-1+i:lags-1+i+t_hf]
            lf_slice = lf_preds[i:i+t_hf]
            # Residual = full signal - LF prediction (this contains HF + noise)
            residual_full = full_slice - lf_slice
            obs_w[i] = residual_full[:, sensor_locs]
            full_w[i] = residual_full

        n_tr = max(1, self.n_train - t_hf + 1)
        n_va = max(1, self.n_valid)
        n_tr = min(n_tr, n_win)
        n_va_end = min(n_tr + n_va, n_win)

        self.hf_train = HFWindowDataset(obs_w[:n_tr], full_w[:n_tr])
        self.hf_valid = HFWindowDataset(obs_w[n_tr:n_va_end], full_w[n_tr:n_va_end])
        self.hf_test = HFWindowDataset(obs_w[n_va_end:], full_w[n_va_end:])
        self.lf_preds_all = lf_preds

        print(f"HF windows: train={len(self.hf_train)}, "
              f"valid={len(self.hf_valid)}, test={len(self.hf_test)}")
        return self


###############################################################################
# GRU LATENT NORMALIZATION (matching original SINDy-SHRED gru_normalize)
###############################################################################

def gru_normalize(z, z_train_ref):
    """Normalize latent trajectories to [-1, 1] using training min/max."""
    z_out = z.clone()
    for d in range(z.shape[1]):
        z_min = z_train_ref[:, d].min()
        z_max = z_train_ref[:, d].max()
        z_out[:, d] = (z_out[:, d] - z_min) / (z_max - z_min + 1e-10)
    z_out = 2.0 * z_out - 1.0
    return z_out


###############################################################################
# TRAINING: LF PATHWAY
###############################################################################

def train_lf(model, train_ds, valid_ds, config, device):
    """Train the LF pathway.

    sindy_shred mode: GRU + SINDy regularization + periodic thresholding.
    vanilla_shred mode: Pure reconstruction MSE loss.
    """
    print("\n" + "="*60)
    print(f"Stage 1: Training LF Pathway  [{config.lf_mode}]")
    print("="*60)

    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=config.lf_lr)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=10, factor=0.5)

    train_ld = DataLoader(train_ds, batch_size=config.lf_batch_size,
                          shuffle=(config.lf_mode != "sindy_shred"))
    valid_ld = DataLoader(valid_ds, batch_size=config.lf_batch_size)

    use_sindy = (config.lf_mode == "sindy_shred")
    dt_sindy = 1.0 / config.fs

    best_loss, best_state, patience_ctr = float('inf'), None, 0
    history = {"train": [], "valid": [], "sindy_loss": []}

    for ep in range(config.lf_epochs):
        model.train()
        tl, sl_accum = 0, 0

        for bx, by in train_ld:
            bx, by = bx.to(device), by.to(device)

            pred = model(bx)
            l_recon = F.mse_loss(pred, by)

            l_sindy = torch.tensor(0.0, device=device)
            if use_sindy and ep >= config.sindy_warmup_epochs:
                # SINDy regularization via Euler integration:
                # Instead of ||dz/dt - predicted_dz/dt||, we use
                # ||z_{i+1} - z_i_predicted_next|| where z_i_predicted_next
                # is obtained by integrating dz/dt forward using sub-stepped
                # Euler method (dt_sub = dt/3 for stability).
                z_batch = model.encode(bx)  # [B, latent_dim]

                if z_batch.shape[0] >= 2:
                    z_current = z_batch[:-1]   # z_i
                    z_next = z_batch[1:]       # z_{i+1} (true next state)

                    # Sub-stepped Euler: integrate from z_i to z_{i+1}
                    n_substeps = 3
                    dt_sub = dt_sindy / n_substeps
                    z_pred_next = z_current
                    for _ in range(n_substeps):
                        dz = model.sindy_predict_dz(z_pred_next)
                        z_pred_next = z_pred_next + dt_sub * dz

                    l_sindy = F.mse_loss(z_pred_next, z_next)

            sindy_weight = 0.0
            if use_sindy and ep >= config.sindy_warmup_epochs:
                # Gradual ramp from 0 to full over remaining epochs
                ramp_length = max(1, config.lf_epochs - config.sindy_warmup_epochs)
                progress = min(1.0, (ep - config.sindy_warmup_epochs) / (ramp_length * 0.5))
                sindy_weight = config.sindy_regularization * progress

            loss = l_recon + sindy_weight * l_sindy

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            tl += l_recon.item() * len(bx)
            sl_accum += l_sindy.item() * len(bx)

        tl /= len(train_ds)
        sl_accum /= len(train_ds)

        # Periodic SINDy STLS: threshold then refit on full training data
        if (use_sindy and ep >= config.sindy_warmup_epochs
                and (ep + 1) % config.sindy_thres_epoch == 0):
            # Step 1: threshold
            model.apply_sindy_threshold(config.sindy_threshold)

            # Step 2: gather full training latent trajectory for refit
            model.eval()
            with torch.no_grad():
                z_all = model.encode(train_ds.X.to(device))   # [N, latent_dim]
                # Central finite difference: dz/dt ≈ (z[i+1] - z[i-1]) / (2*dt)
                if z_all.shape[0] >= 3:
                    dz_dt_all = (z_all[2:] - z_all[:-2]) / (2.0 * dt_sindy)
                    z_mid_all = z_all[1:-1]
                    # Step 3: least-squares refit on active support
                    model.stls_refit(z_mid_all, dz_dt_all)
            model.train()

        model.eval()
        vl = 0
        with torch.no_grad():
            for bx, by in valid_ld:
                bx, by = bx.to(device), by.to(device)
                vl += F.mse_loss(model(bx), by).item() * len(bx)
        vl /= len(valid_ds)

        sched.step(vl)
        history["train"].append(tl)
        history["valid"].append(vl)
        history["sindy_loss"].append(sl_accum)

        if vl < best_loss:
            best_loss = vl
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1

        if (ep+1) % 50 == 0 or ep == 0:
            sindy_str = f" sindy={sl_accum:.6f}" if use_sindy else ""
            print(f"  Epoch {ep+1:4d}: train={tl:.6f} valid={vl:.6f}"
                  f" (best={best_loss:.6f}){sindy_str}")
        if patience_ctr >= config.lf_patience:
            print(f"  Early stopping at epoch {ep+1}")
            break

    model.load_state_dict(best_state)
    model = model.to(device)
    print(f"  Best validation loss: {best_loss:.6f}")

    if use_sindy:
        Xi = (model.sindy_coefficients * model.sindy_mask).detach().cpu().numpy()
        n_active = int((np.abs(Xi) > 1e-10).sum())
        print(f"  SINDy coefficients: {n_active}/{Xi.size} active terms")

    return history


###############################################################################
# POST-HOC SINDY IDENTIFICATION
###############################################################################

def sindy_posthoc_identify(model, train_ds, config, device):
    """Post-hoc SINDy identification on learned latent space.

    Following the reference SINDy-SHRED implementation:
    1. Extract normalized GRU latent trajectories from training data
    2. Fit a pysindy model using STLSQ with the configured threshold
    3. Test stability by simulation with timeout protection
    4. Fall back to unthresholded least-squares if unstable

    Returns:
        z_normed_np: normalized latent from GRU  [T, latent_dim]
        z_sim: SINDy-identified model simulation  [T, latent_dim]
        sindy_info: dict with coefficient info
    """
    import pysindy as ps
    from scipy.integrate import solve_ivp

    model.eval()
    dt_sindy = 1.0 / config.fs

    # --- Step 1: Extract normalized GRU latent trajectories ---
    with torch.no_grad():
        all_z = model.encode(train_ds.X.to(device))

    z_normed = gru_normalize(all_z, all_z)
    z_np = z_normed.detach().cpu().numpy()
    T = len(z_np)
    t_sim = np.arange(T) * dt_sindy

    diff_method = ps.differentiation.SmoothedFiniteDifference()
    feature_lib = ps.PolynomialLibrary(degree=config.sindy_poly_order)

    # --- Step 2: Fit STLSQ with the user-configured threshold ---
    threshold = config.sindy_posthoc_threshold

    sindy_model = ps.SINDy(
        optimizer=ps.STLSQ(threshold=threshold, alpha=0.7),
        differentiation_method=diff_method,
        feature_library=feature_lib,
    )
    sindy_model.fit(z_np, t=dt_sindy)

    n_nonzero = np.count_nonzero(sindy_model.coefficients())
    print(f"  Post-hoc SINDy: threshold={threshold:.4f}, "
          f"{n_nonzero}/{sindy_model.coefficients().size} active terms")

    divergence_limit = 10.0  # latent space is normalized to [-1, 1]

    def _try_simulate(candidate_model, z0, t_eval, max_val=divergence_limit):
        """Simulate with solve_ivp and a hard max_step to avoid hanging."""
        try:
            def rhs(t, z):
                return candidate_model.predict(z.reshape(1, -1)).flatten()

            sol = solve_ivp(
                rhs, (t_eval[0], t_eval[-1]), z0,
                t_eval=t_eval, method='RK45',
                rtol=1e-6, atol=1e-8,
                max_step=dt_sindy * 10,
            )

            if sol.status != 0 or sol.y.shape[1] < len(t_eval):
                return None
            z_out = sol.y.T
            if np.any(np.abs(z_out) > max_val) or np.any(np.isnan(z_out)):
                return None
            return z_out
        except Exception:
            return None

    # --- Step 3: Simulate and check stability ---
    z_sim = _try_simulate(sindy_model, z_np[0], t_sim)

    if z_sim is not None:
        mse = float(np.mean((z_sim - z_np) ** 2))
        best_model = sindy_model
        best_threshold = threshold
    else:
        # Fallback: unthresholded least-squares
        print("  WARNING: STLSQ model diverged. Falling back to "
              "unthresholded least-squares fit.")
        ls_model = ps.SINDy(
            optimizer=ps.STLSQ(threshold=0.0, alpha=1.0),
            differentiation_method=diff_method,
            feature_library=feature_lib,
        )
        ls_model.fit(z_np, t=dt_sindy)
        z_sim = _try_simulate(ls_model, z_np[0], t_sim)
        if z_sim is None:
            print("  WARNING: Fallback simulation also diverged, "
                  "filling with NaN.")
            z_sim = np.full_like(z_np, np.nan)
            mse = float('nan')
        else:
            mse = float(np.mean((z_sim - z_np) ** 2))
        best_model = ls_model
        best_threshold = 0.0

    Xi_posthoc = best_model.coefficients()
    n_active = int((np.abs(Xi_posthoc) > 1e-10).sum())
    n_total = Xi_posthoc.size

    print(f"  Selected threshold={best_threshold:.4f}, "
          f"{n_active}/{n_total} active, MSE={mse:.4e}")
    print("  Discovered equations:")
    best_model.print()

    sindy_info = {
        "coefficients": Xi_posthoc,
        "n_active": n_active,
        "n_total": n_total,
        "threshold": best_threshold,
        "mse": mse,
        "model": best_model,
    }

    return z_np, z_sim, sindy_info


###############################################################################
# TRAINING: HF LAYERS
###############################################################################

def train_hf_layer(full_model, layer_idx, train_ds, valid_ds, config, device):
    """Train one HF peeling layer."""
    print(f"\n  --- HF Peel Layer {layer_idx+1} ---")

    hf_layer = full_model.hf_layers[layer_idx]
    sensor_idx = full_model.sensor_indices

    for p in full_model.parameters():
        p.requires_grad = False
    for p in hf_layer.parameters():
        p.requires_grad = True

    opt = torch.optim.Adam(hf_layer.parameters(), lr=config.hf_lr)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=10, factor=0.5)

    train_ld = DataLoader(train_ds, batch_size=config.hf_batch_size, shuffle=True)
    valid_ld = DataLoader(valid_ds, batch_size=config.hf_batch_size)

    best_loss, best_state, pat = float('inf'), None, 0
    history = {"train": [], "valid": [], "sparsity": []}
    total_ep = config.hf_epochs + config.finetune_epochs

    for ep in range(total_ep):
        if ep < config.hf_warmup_epochs:
            lam = 0.0
        elif ep < config.hf_epochs:
            progress = (ep - config.hf_warmup_epochs) / max(
                1, config.hf_epochs - config.hf_warmup_epochs)
            lam = config.lambda_sparse * progress
        else:
            lam = config.lambda_sparse_finetune

        hf_layer.train()
        tl, sl, ns = 0, 0, 0
        for obs_w, full_w in train_ld:
            obs_w, full_w = obs_w.to(device), full_w.to(device)
            corr = hf_layer(obs_w)

            l_sensor = F.mse_loss(corr[:, :, sensor_idx], obs_w)
            l_full = F.mse_loss(corr, full_w)
            l_recon = 0.5 * l_sensor + 0.5 * l_full
            l_sparse = temporal_freq_sparsity(corr, config.fs, config.max_freq_hz)
            l_mag = mag_penalty(corr, config.mag_threshold)
            loss = l_recon + lam * l_sparse + config.lambda_mag * l_mag

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(hf_layer.parameters(), 1.0)
            opt.step()
            tl += l_recon.item() * len(obs_w)
            sl += l_sparse.item() * len(obs_w)
            ns += len(obs_w)
        tl /= ns
        sl /= ns

        hf_layer.eval()
        vl = 0
        with torch.no_grad():
            for obs_w, full_w in valid_ld:
                obs_w, full_w = obs_w.to(device), full_w.to(device)
                vl += F.mse_loss(hf_layer(obs_w), full_w).item() * len(obs_w)
        vl /= max(1, len(valid_ds))

        sched.step(vl)
        history["train"].append(tl)
        history["valid"].append(vl)
        history["sparsity"].append(sl)

        if vl < best_loss:
            best_loss = vl
            best_state = {k: v.cpu().clone() for k, v in hf_layer.state_dict().items()}
            pat = 0
        else:
            pat += 1

        if (ep+1) % 50 == 0 or ep == 0:
            phase = ("warm" if ep < config.hf_warmup_epochs
                     else ("main" if ep < config.hf_epochs else "fine"))
            print(f"    Ep {ep+1:4d} [{phase}]: recon={tl:.6f} sparse={sl:.4f} "
                  f"lam={lam:.4f} val={vl:.6f} gamma={hf_layer.gamma.item():.4f}")

        if pat >= config.hf_patience and ep > config.hf_warmup_epochs:
            print(f"    Early stopping at epoch {ep+1}")
            break

    if best_state:
        hf_layer.load_state_dict(best_state)
    for p in full_model.parameters():
        p.requires_grad = True
    print(f"    Best valid loss: {best_loss:.6f}")
    return history

###############################################################################
# EVALUATION & VISUALIZATION
###############################################################################


def evaluate_and_plot(full_model, data_prep, config, device, output_dir):
    """Run evaluation and save plots.

    Reconstruction comparison plots use UNSCALED (original Hz deviation)
    so that HF oscillations and noise are visible at their natural scale.
    """
    output_dir = Path(output_dir)
    plot_dir = output_dir / "plots"
    plot_dir.mkdir(parents=True, exist_ok=True)

    full_model.eval()
    sensor_idx = full_model.sensor_indices
    scaler = data_prep.scaler

    # --- LF metrics ---
    # LF targets are smoothed; compare LF preds against smoothed targets
    all_lf_targets_scaled = torch.cat([data_prep.lf_train.Y, data_prep.lf_valid.Y,
                                       data_prep.lf_test.Y]).numpy()
    lf_preds_scaled = data_prep.lf_preds_all
    lf_rmse_scaled = np.sqrt(np.mean((lf_preds_scaled - all_lf_targets_scaled)**2))
    print(f"\n  LF RMSE (scaled, vs smoothed target): {lf_rmse_scaled:.6f}")

    # Full (unsmoothed) scaled signal as ground truth for reconstruction plots
    lags = config.lags
    n_usable = len(lf_preds_scaled)
    all_full_scaled = data_prep.x_scaled[lags-1:lags-1+n_usable]

    # --- Unscaled versions for plotting ---
    n_plot = min(500, len(data_prep.lf_test))
    test_start = data_prep.n_train + data_prep.n_valid
    test_x = data_prep.lf_test.X[:n_plot].to(device)
    # Ground truth = full (unsmoothed) signal
    test_y_scaled = all_full_scaled[test_start:test_start+n_plot]
    with torch.no_grad():
        lf_pred_scaled = full_model.lf_model(test_x).cpu().numpy()

    test_y_unscaled = scaler.inverse_transform(test_y_scaled)
    lf_pred_unscaled = scaler.inverse_transform(lf_pred_scaled)

    lf_rmse_unscaled = np.sqrt(np.mean((lf_pred_unscaled - test_y_unscaled)**2))
    print(f"  LF RMSE (unscaled, Hz): {lf_rmse_unscaled:.6f}")

    # --- HF metrics on test ---
    test_ld = DataLoader(data_prep.hf_test, batch_size=64)
    all_corr, all_tgt = [], []
    with torch.no_grad():
        for obs_w, full_w in test_ld:
            obs_w = obs_w.to(device)
            corr, _ = full_model.forward_hf(obs_w)
            all_corr.append(corr.cpu().numpy())
            all_tgt.append(full_w.numpy())

    hf_rmse, improvement = 0.0, 0.0
    has_hf = len(all_corr) > 0 and all_corr[0].size > 0
    if has_hf:
        corrs = np.vstack(all_corr)
        tgts = np.vstack(all_tgt)
        hf_rmse = np.sqrt(np.mean((corrs - tgts)**2))
        resid_mag = np.sqrt(np.mean(tgts**2))
        improvement = (resid_mag - hf_rmse) / resid_mag * 100
        print(f"  HF RMSE (test):   {hf_rmse:.6f}")
        print(f"  Residual before:  {resid_mag:.6f}")
        print(f"  Improvement:      {improvement:.1f}%")

    M = config.n_pmu_total
    obs = list(data_prep.sensor_locs)
    unobs = [k for k in range(M) if k not in obs]

    # pmus_to_plot = [obs[0], obs[-1], unobs[0], unobs[-1]]
    # pmu_labels = ["Observed", "Observed", "Unobserved", "Unobserved"]

    pmus_to_plot = obs + unobs
    pmu_labels = ["Observed"] * len(obs) + ["Unobserved"] * len(unobs)

    # =================================================================
    # Plot 1: Side-by-side  LF-only vs Full (LF+HF) reconstruction
    #         in UNSCALED Hz-deviation space — FULL time sequence
    # =================================================================

   
    # --- Build per-timestep HF corrections for all datasets ---
    t_hf = config.t_hf
    print(f"t_hf={t_hf}")
    n_total_usable = len(lf_preds_scaled)
    print(f"Total reconstruction windows: {n_total_usable}")

    hf_correction_scaled = np.zeros((n_total_usable, M))
    hf_count = np.zeros(n_total_usable)


    if has_hf:
            n_win_total = n_total_usable - t_hf + 1
            n_tr_hf = max(1, data_prep.n_train - t_hf + 1)
            n_va_hf = max(1, data_prep.n_valid)
            n_va_end_hf = min(n_tr_hf + n_va_hf, n_win_total)
            hf_test_start_idx = n_va_end_hf

            # Accumulate HF corrections for ALL splits (train, valid, test)
            all_hf_splits = [
                (data_prep.hf_train_orig, 0),
                (data_prep.hf_valid_orig, n_tr_hf),
                (data_prep.hf_test, hf_test_start_idx),
            ]
            with torch.no_grad():
                for hf_ds, start_idx in all_hf_splits:
                    hf_ld = DataLoader(hf_ds, batch_size=64)
                    win_idx = start_idx
                    for obs_w, _ in hf_ld:
                        obs_w = obs_w.to(device)
                        corr, _ = full_model.forward_hf(obs_w)
                        corr_np = corr.cpu().numpy()
                        for b in range(len(corr_np)):
                            for t_off in range(t_hf):
                                idx = win_idx + b + t_off
                                if idx < n_total_usable:
                                    hf_correction_scaled[idx] += corr_np[b, t_off]
                                    hf_count[idx] += 1
                        win_idx += len(corr_np)


            # Average corrections where we have contributions
            valid_mask = hf_count > 0
            hf_correction_scaled[valid_mask] /= hf_count[valid_mask, None]

            # Optionally, save valid_mask to CSV for debug
            count_df = pd.DataFrame(valid_mask.astype(int), columns=["count"])
            count_df.to_csv(output_dir / "valid_mask.csv", index=False)

    print("HF corrections computed for all datasets (train+valid+test).")

    # Full reconstruction = LF + HF (both in scaled space)
    full_recon_scaled = lf_preds_scaled + hf_correction_scaled

    # save csv
    recon_df = pd.DataFrame(hf_correction_scaled, columns=[f"PMU_{i}" for i in range(M)])
    recon_df.to_csv(output_dir / "hf_correction_scaled.csv", index=False)
    
    # Plot FULL train+valid+test sequence (unscaled)

    lf_all_unscaled = scaler.inverse_transform(lf_preds_scaled)
    full_all_unscaled = scaler.inverse_transform(full_recon_scaled)
    gt_all_unscaled = scaler.inverse_transform(all_full_scaled)


    # Time axis in seconds
    dt_plot = 1.0 / config.fs
    t_recon = np.arange(n_total_usable) * dt_plot

    # Split boundaries in seconds
    t_train_end = data_prep.n_train * dt_plot
    t_valid_end = (data_prep.n_train + data_prep.n_valid) * dt_plot

    # =================================================================
    # Prepare SINDy-based inference extension for LF panel (if available)
    # =================================================================
    use_sindy = (config.lf_mode == "sindy_shred")
    has_inference = (data_prep.inference_truth_raw is not None and use_sindy)

    infer_truth_unscaled = None
    infer_pred_unscaled = None
    n_infer = 0
    t_infer_boundary = t_recon[-1]  # reconstruction-inference boundary in seconds
    sindy_info_for_infer = None
    z_shred_for_infer = None

    if has_inference:
        from scipy.integrate import solve_ivp

        infer_truth_unscaled = data_prep.inference_truth_raw  # raw CSV values
        n_infer = len(infer_truth_unscaled)

        # We need the post-hoc SINDy model — do identification first if not done yet
        # (It will be done again in the latent space section, but we need it here)
        z_shred_for_infer, _, sindy_info_for_infer = sindy_posthoc_identify(
            full_model.lf_model, data_prep.lf_train, config, device)

        sindy_model = sindy_info_for_infer.get("model")
        if sindy_model is not None:
            # Get the last latent state from the reconstruction window
            # Encode all reconstruction data and normalize
            all_X_recon = torch.cat([data_prep.lf_train.X, data_prep.lf_valid.X,
                                     data_prep.lf_test.X])
            with torch.no_grad():
                z_all_recon = full_model.lf_model.encode(all_X_recon.to(device))
                z_train_ref = full_model.lf_model.encode(
                    data_prep.lf_train.X.to(device))
            z_all_normed = gru_normalize(z_all_recon, z_train_ref)
            z_last = z_all_normed[-1].detach().cpu().numpy()

            # Roll forward using SINDy ODE in normalized latent space
            t_infer = np.arange(n_infer + 1) * dt_plot  # +1 to include start
            try:
                def rhs_infer(t, z):
                    return sindy_model.predict(z.reshape(1, -1)).flatten()
                sol = solve_ivp(rhs_infer, (0, t_infer[-1]), z_last,
                                t_eval=t_infer, method='RK45',
                                rtol=1e-6, atol=1e-8,
                                max_step=dt_plot * 10)
                if sol.status == 0 and sol.y.shape[1] >= n_infer + 1:
                    z_infer_normed = sol.y.T[1:]  # skip the initial state (t=0)
                else:
                    z_infer_normed = None
            except Exception:
                z_infer_normed = None

            if z_infer_normed is not None:
                # Denormalize: reverse of gru_normalize
                # gru_normalize does: z_out = 2 * (z - min) / (max - min) - 1
                # inverse: z = (z_normed + 1) / 2 * (max - min) + min
                z_train_np = z_train_ref.detach().cpu()
                z_denormed = torch.zeros(n_infer, config.latent_dim)
                z_infer_t = torch.tensor(z_infer_normed, dtype=torch.float32)
                for d in range(config.latent_dim):
                    z_min = z_train_np[:, d].min()
                    z_max = z_train_np[:, d].max()
                    z_denormed[:, d] = (z_infer_t[:, d] + 1.0) / 2.0 * (z_max - z_min + 1e-10) + z_min

                # Decode through frozen decoder to get physical-space prediction
                with torch.no_grad():
                    infer_pred_scaled = full_model.lf_model.decode(
                        z_denormed.to(device)).cpu().numpy()

                # Inverse-transform to unscaled Hz
                infer_pred_unscaled = scaler.inverse_transform(infer_pred_scaled)
            else:
                has_inference = False
                print("  WARNING: SINDy inference rollout failed/diverged.")
        else:
            has_inference = False

    fig, axes = plt.subplots(len(pmus_to_plot), 2,
                             figsize=(20, 3 * len(pmus_to_plot)),
                             sharex='col')

    for row, (s, label) in enumerate(zip(pmus_to_plot, pmu_labels)):
        # Left column: LF-only reconstruction + inference extension
        ax_lf = axes[row, 0]
        ax_lf.plot(t_recon, gt_all_unscaled[:, s], 'k', lw=0.7, alpha=0.7, label="Truth")
        ax_lf.plot(t_recon, lf_all_unscaled[:, s], 'b--', lw=0.7, label="LF pred (recon)")

        if has_inference:
            # Extend time axis for inference portion
            t_infer_plot = t_recon[-1] + np.arange(1, n_infer + 1) * dt_plot
            ax_lf.plot(t_infer_plot, infer_truth_unscaled[:, s], 'k', lw=0.7, alpha=0.7)
            ax_lf.plot(t_infer_plot, infer_pred_unscaled[:, s], 'b--', lw=0.7,
                       alpha=0.7, label="LF pred (SINDy inference)")
            # Purple vertical line at reconstruction-inference boundary
            ax_lf.axvline(t_infer_boundary, color='purple', ls='-', lw=1.2, alpha=0.8)

        ax_lf.axvline(t_train_end, color='gray', ls=':', lw=0.8, alpha=0.6)
        ax_lf.axvline(t_valid_end, color='gray', ls=':', lw=0.8, alpha=0.6)
        ax_lf.set_ylabel(f"PMU {s}\n[Hz dev]")
        if row == 0:
            ax_lf.set_title("LF-only Reconstruction + SINDy Inference")
            ax_lf.legend(fontsize=6, loc='upper right')
        if row == len(pmus_to_plot) - 1:
            ax_lf.set_xlabel("Time [s]")
        ax_lf.annotate(f"{label}", xy=(0.01, 0.95), xycoords='axes fraction',
                       fontsize=7, va='top', color='gray')

        # Right column: Full (LF+HF) reconstruction (UNCHANGED)
        ax_full = axes[row, 1]
        ax_full.plot(t_recon, gt_all_unscaled[:, s], 'k', lw=0.7, alpha=0.7, label="Truth")
        ax_full.plot(t_recon, full_all_unscaled[:, s], 'r--', lw=0.7, label="LF+HF pred")


        ax_full.axvline(t_train_end, color='gray', ls=':', lw=0.8, alpha=0.6)
        ax_full.axvline(t_valid_end, color='gray', ls=':', lw=0.8, alpha=0.6)

        ax_full.set_ylabel(f"PMU {s}\n[Hz dev]")
        if row == 0:
            ax_full.set_title("Full (LF+HF) Reconstruction")
            ax_full.legend(fontsize=7)
        if row == len(pmus_to_plot) - 1:
            ax_full.set_xlabel("Time [s]")
        ax_full.annotate(f"{label}", xy=(0.01, 0.95), xycoords='axes fraction',
                         fontsize=7, va='top', color='gray')

    # Add split labels at top of left panel
    axes[0, 0].annotate("train", xy=(t_train_end * 0.4, 1.02),
                         xycoords=("data", "axes fraction"), fontsize=7, color='gray')
    axes[0, 0].annotate("val", xy=((t_train_end + t_valid_end) * 0.5, 1.02),
                         xycoords=("data", "axes fraction"), fontsize=7, color='gray')
    if has_inference:
        axes[0, 0].annotate("test", xy=((t_valid_end + t_infer_boundary) * 0.5, 1.02),
                             xycoords=("data", "axes fraction"), fontsize=7, color='gray')
        axes[0, 0].annotate("inference", xy=((t_infer_boundary + axes[0,0].get_xlim()[1]) * 0.5, 1.02),
                             xycoords=("data", "axes fraction"), fontsize=7, color='purple')
    else:
        axes[0, 0].annotate("test", xy=((t_valid_end + t_recon[-1]) * 0.5, 1.02),
                             xycoords=("data", "axes fraction"), fontsize=7, color='gray')
    # Split labels for right panel
    axes[0, 1].annotate("train", xy=(t_train_end * 0.4, 1.02),
                         xycoords=("data", "axes fraction"), fontsize=7, color='gray')
    axes[0, 1].annotate("val", xy=((t_train_end + t_valid_end) * 0.5, 1.02),
                         xycoords=("data", "axes fraction"), fontsize=7, color='gray')
    axes[0, 1].annotate("test", xy=((t_valid_end + t_recon[-1]) * 0.5, 1.02),
                         xycoords=("data", "axes fraction"), fontsize=7, color='gray')

    fig.suptitle("Reconstruction Comparison (unscaled Hz deviation)", fontsize=12)
    fig.tight_layout()
    fig.savefig(plot_dir / "01_reconstruction_comparison.png", dpi=150)

    
    ### ANDREA SALVARE CSV ###

    # Base reconstruction dataframe
    data_dict = {
        "time": t_recon
    }

    for s in pmus_to_plot:
        data_dict[f"GT_PMU_{s}"] = gt_all_unscaled[:, s]
        data_dict[f"LF_PMU_{s}"] = lf_all_unscaled[:, s]
        data_dict[f"LFHF_PMU_{s}"] = full_all_unscaled[:, s]

    df_recon = pd.DataFrame(data_dict)

    if has_inference:
        # Build inference time axis
        t_infer_plot = t_recon[-1] + np.arange(1, n_infer + 1) * dt_plot

        infer_dict = {
            "time": t_infer_plot
        }

        for s in pmus_to_plot:
            infer_dict[f"GT_PMU_{s}"] = infer_truth_unscaled[:, s]
            infer_dict[f"LF_PMU_{s}"] = infer_pred_unscaled[:, s]
            infer_dict[f"LFHF_PMU_{s}"] = np.nan  # HF not defined during inference

        df_infer = pd.DataFrame(infer_dict)

        # Concatenate reconstruction + inference
        df_export = pd.concat([df_recon, df_infer], ignore_index=True)

    else:
        df_export = df_recon

    # Save CSV
    csv_path = plot_dir / "01_reconstruction_comparison.csv"
    df_export.to_csv(csv_path, index=False)
    plt.close(fig)

    # =================================================================
    # Plot 2: HF correction spectrum
    # =================================================================
    if has_hf:
        corrs_spec = np.vstack(all_corr)
        n_w, t_hf_c, M_c = corrs_spec.shape
        freqs = np.fft.rfftfreq(t_hf_c, d=1.0/config.fs)
        avg_spec = np.mean(np.abs(np.fft.rfft(corrs_spec, axis=1)), axis=(0, 2))

        fig, ax = plt.subplots(figsize=(10, 4))
        ax.bar(freqs, avg_spec,
               width=freqs[1]-freqs[0] if len(freqs) > 1 else 0.1,
               alpha=0.7, color='steelblue')
        ax.axvline(config.max_freq_hz, color='red', ls='--',
                   label=f'max_freq={config.max_freq_hz}')
        ax.set_xlabel("Frequency [Hz]")
        ax.set_ylabel("Mean |FFT|")
        ax.set_title("HF correction average spectrum")
        ax.legend()
        fig.tight_layout()
        fig.savefig(plot_dir / "02_hf_spectrum.png", dpi=150)
        plt.close(fig)

    # =================================================================
    # Plot 3: Latent space
    #   sindy_shred mode -> SINDy-SHRED latent vs identified model
    #                       (matching collaborator's latent.jpg)
    #   vanilla_shred mode -> learned latent vs GT latent (best effort)
    # =================================================================
    use_sindy = (config.lf_mode == "sindy_shred")

    if use_sindy:
        # Reuse sindy_info if already computed for inference extension
        if has_inference and sindy_info_for_infer is not None:
            sindy_info = sindy_info_for_infer
            z_shred = z_shred_for_infer
        else:
            z_shred, _, sindy_info = sindy_posthoc_identify(
                full_model.lf_model, data_prep.lf_train, config, device)

        n_active = sindy_info["n_active"]
        n_total_coeffs = sindy_info["n_total"]
        print(f"  Post-hoc SINDy: {n_active}/{n_total_coeffs} active terms")

        # --- Encode ALL data (train+valid+test) for full visualization ---
        all_X = torch.cat([data_prep.lf_train.X, data_prep.lf_valid.X,
                           data_prep.lf_test.X])
        with torch.no_grad():
            all_z = full_model.lf_model.encode(all_X.to(device))
        # Normalize using TRAINING stats (same as sindy_posthoc_identify)
        z_train = full_model.lf_model.encode(
            data_prep.lf_train.X.to(device)).detach()
        z_all_normed = gru_normalize(all_z, z_train)
        z_all_np = z_all_normed.detach().cpu().numpy()

        # Simulate SINDy ODE over the full time range
        from scipy.integrate import solve_ivp
        dt_plot = 1.0 / config.fs
        t_full = np.arange(len(z_all_np)) * dt_plot
        try:
            def rhs(t, z):
                return sindy_info["model"].predict(z.reshape(1, -1)).flatten()
            sol = solve_ivp(rhs, (t_full[0], t_full[-1]), z_all_np[0],
                            t_eval=t_full, method='RK45',
                            rtol=1e-6, atol=1e-8,
                            max_step=dt_plot * 10)
            z_sim_full = sol.y.T if sol.status == 0 else np.full_like(z_all_np, np.nan)
        except Exception:
            z_sim_full = np.full_like(z_all_np, np.nan)

        # Split boundaries in seconds
        t_train_end = data_prep.n_train * dt_plot
        t_valid_end = (data_prep.n_train + data_prep.n_valid) * dt_plot

        fig, axes = plt.subplots(config.latent_dim, 1,
                                 figsize=(12, 2.5 * config.latent_dim),
                                 sharex=True)
        if config.latent_dim == 1:
            axes = [axes]
        for i in range(config.latent_dim):
            axes[i].plot(t_full, z_all_np[:, i], 'b', lw=0.8,
                         label='GRU latent (encoder output)')
            axes[i].plot(t_full, z_sim_full[:, i], 'k--', lw=0.8,
                         label='SINDy identified ODE')
            axes[i].axvline(t_train_end, color='gray', ls=':', lw=0.8, alpha=0.6)
            axes[i].axvline(t_valid_end, color='gray', ls=':', lw=0.8, alpha=0.6)
            axes[i].set_ylabel(f"$z_{i}$")
            if i == config.latent_dim - 1:
                axes[i].legend(fontsize=8)
        # Split labels
        axes[0].annotate("train", xy=(t_train_end * 0.4, 1.02),
                         xycoords=("data", "axes fraction"), fontsize=7, color='gray')
        axes[0].annotate("val", xy=((t_train_end + t_valid_end) * 0.5, 1.02),
                         xycoords=("data", "axes fraction"), fontsize=7, color='gray')
        axes[0].annotate("test", xy=((t_valid_end + t_full[-1]) * 0.5, 1.02),
                         xycoords=("data", "axes fraction"), fontsize=7, color='gray')
        axes[-1].set_xlabel("Time [s]")
        axes[0].set_title("Latent Space: GRU Encoder vs SINDy Identified Model")
        fig.tight_layout()
        fig.savefig(plot_dir / "03_latent_space.png", dpi=150)

        # ===========================
        # CREA DATAFRAME UNICO
        # ===========================
        data_dict = {"time": t_full}

        for i in range(config.latent_dim):
            data_dict[f"z{i}_all"] = z_all_np[:, i]
            data_dict[f"z{i}_sim"] = z_sim_full[:, i]

        df_latent = pd.DataFrame(data_dict)

        # ===========================
        # SALVA CSV
        # ===========================
        csv_path = plot_dir / "latent_space_comparison.csv"
        df_latent.to_csv(csv_path, index=False)
        plt.close(fig)

    else:
        # Vanilla SHRED: compare learned latent vs GT latent
        all_X = data_prep.lf_train.X
        with torch.no_grad():
            z_pred = full_model.lf_model.encode(all_X.to(device)).cpu().numpy()
        z_gt = data_prep.gt_latent[config.lags-1:config.lags-1+len(z_pred)]

        fig, axes = plt.subplots(config.latent_dim, 1,
                                 figsize=(14, 2*config.latent_dim), sharex=True)
        if config.latent_dim == 1:
            axes = [axes]
        for i in range(config.latent_dim):
            z_i = z_pred[:, i]
            z_i_norm = 2*(z_i - z_i.min())/(z_i.max()-z_i.min()+1e-10) - 1
            gt_i = z_gt[:len(z_i_norm), i]
            gt_i_norm = 2*(gt_i - gt_i.min())/(gt_i.max()-gt_i.min()+1e-10) - 1
            axes[i].plot(z_i_norm, 'b', lw=0.7, label='SHRED latent')
            axes[i].plot(gt_i_norm, 'k--', lw=0.7, alpha=0.5, label='GT latent')
            axes[i].set_ylabel(f"$z_{i}$")
            if i == 0:
                axes[i].legend(fontsize=7)
        axes[-1].set_xlabel("Time step")
        axes[0].set_title("Learned vs Ground Truth Latent Space")
        fig.tight_layout()
        fig.savefig(plot_dir / "03_latent_space.png", dpi=150)
        plt.close(fig)

    # =================================================================
    # Save metrics
    # =================================================================
    metrics = {
        "lf_rmse_scaled": float(lf_rmse_scaled),
        "lf_rmse_unscaled_hz": float(lf_rmse_unscaled),
        "hf_rmse_test": float(hf_rmse) if has_hf else None,
        "hf_improvement_pct": float(improvement) if has_hf else None,
        "lf_mode": config.lf_mode,
    }
    if use_sindy:
        metrics["sindy_active_terms"] = sindy_info["n_active"]
        metrics["sindy_total_terms"] = sindy_info["n_total"]

    with open(output_dir / "metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)

    print(f"  Plots and metrics saved to {output_dir}")
    return metrics


###############################################################################
# MAIN PIPELINE
###############################################################################

def run_pipeline(config: PipelineConfig):
    """Execute the full LF + HF pipeline."""
    np.random.seed(config.seed)
    torch.manual_seed(config.seed)
    device = config.resolve_device()
    print(f"Device: {device}")
    print(f"LF mode: {config.lf_mode}")

    output_dir = Path(config.data_dir) / "results"
    output_dir.mkdir(parents=True, exist_ok=True)

    # Auto-save all console output
    from datetime import datetime
    log_path = output_dir / f"log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
    tee = TeeLogger(str(log_path))
    sys.stdout = tee

    # --- Load data ---
    prep = DataPreparer(config).load_and_prepare()

    # --- Build LF model ---
    lf_model = SHREDNet(
        n_sensors=config.n_pmu_observed,
        state_dim=config.n_pmu_total,
        latent_dim=config.latent_dim,
        hidden_size=config.lstm_hidden,
        n_layers=config.lstm_layers,
        decoder_hidden=config.decoder_hidden,
        dropout=config.dropout,
        mode=config.lf_mode,
        sindy_poly_order=config.sindy_poly_order,
        sindy_include_sine=config.sindy_include_sine,
    )
    n_params = sum(p.numel() for p in lf_model.parameters())
    print(f"LF model parameters: {n_params:,}")
    if config.lf_mode == "sindy_shred":
        lib_dim = sindy_library_size(config.latent_dim, config.sindy_poly_order,
                                      config.sindy_include_sine)
        print(f"  SINDy library size: {lib_dim}")

    # --- Train LF ---
    lf_hist = train_lf(lf_model, prep.lf_train, prep.lf_valid, config, device)

    # --- Prepare HF data ---
    prep.prepare_hf_windows(lf_model, device)

    # --- Build full model ---
    full_model = FullReconstructionModel(
        lf_model=lf_model,
        sensor_indices=prep.sensor_locs,
        n_total=config.n_pmu_total,
        t_hf=config.t_hf,
        n_peel_layers=config.n_peel_layers,
        hf_conv_channels=config.hf_conv_channels,
        hf_conv_kernel=config.hf_conv_kernel,
        hf_mlp_hidden=config.hf_mlp_hidden,
        dropout=config.dropout,
    ).to(device)

    hf_params = sum(p.numel() for p in full_model.hf_layers.parameters())
    print(f"HF layers parameters: {hf_params:,}")

    # --- Train HF layers sequentially ---
    print("\n" + "="*60)
    print("Stage 3: Training HF Peeling Layers")
    print("="*60)

    hf_hists = []

    # Save original HF datasets before peeling mutates them
    prep.hf_train_orig = HFWindowDataset(
        prep.hf_train.X.numpy().copy(), prep.hf_train.Y.numpy().copy())
    prep.hf_valid_orig = HFWindowDataset(
        prep.hf_valid.X.numpy().copy(), prep.hf_valid.Y.numpy().copy())
    
    for i in range(config.n_peel_layers):
        h = train_hf_layer(full_model, i, prep.hf_train, prep.hf_valid,
                           config, device)
        hf_hists.append(h)

        if i < config.n_peel_layers - 1:
            _update_hf_residuals(full_model, prep, i+1, config, device)

    # --- Evaluate ---
    metrics = evaluate_and_plot(full_model, prep, config, device, output_dir)

    # --- Save checkpoint ---
    torch.save({
        'full_model': full_model.state_dict(),
        'config': vars(config),
        'sensor_locs': prep.sensor_locs,
        'lf_history': lf_hist,
        'hf_histories': hf_hists,
    }, output_dir / 'checkpoint.pt')

    # --- Save training curves ---
    n_curve_panels = 1 + config.n_peel_layers
    fig, axes = plt.subplots(1, n_curve_panels,
                             figsize=(5 * n_curve_panels, 4))
    if n_curve_panels == 1:
        axes = [axes]

    axes[0].plot(lf_hist["train"], label="train")
    axes[0].plot(lf_hist["valid"], label="valid")
    axes[0].set_title(f"LF Training [{config.lf_mode}]")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()
    axes[0].set_yscale('log')

    if config.lf_mode == "sindy_shred" and any(
            v > 0 for v in lf_hist["sindy_loss"]):
        ax_sindy = axes[0].twinx()
        ax_sindy.plot(lf_hist["sindy_loss"], 'g--', alpha=0.5, label="SINDy loss")
        ax_sindy.set_ylabel("SINDy loss", color='green')
        ax_sindy.tick_params(axis='y', labelcolor='green')

    for i, h in enumerate(hf_hists):
        axes[i+1].plot(h["train"], label="train")
        axes[i+1].plot(h["valid"], label="valid")
        axes[i+1].set_title(f"HF Layer {i+1}")
        axes[i+1].set_xlabel("Epoch")
        axes[i+1].legend()
        axes[i+1].set_yscale('log')

    fig.tight_layout()
    fig.savefig(output_dir / "plots" / "00_training_curves.png", dpi=150)
    plt.close(fig)

    print(f"\nAll outputs saved to {output_dir}")
    print("DONE!")

    tee.close()

    return full_model, prep, metrics


def _update_hf_residuals(full_model, prep, n_active, config, device):
    """Recompute HF residuals after training layers 0..n_active-1."""
    full_model.eval()
    loader = DataLoader(prep.hf_train, batch_size=64)
    new_obs, new_full = [], []
    with torch.no_grad():
        for obs_w, full_w in loader:
            obs_w = obs_w.to(device)
            total_corr, _ = full_model.forward_hf(obs_w, n_layers=n_active)
            new_full_w = full_w.to(device) - total_corr
            new_obs_w = new_full_w[:, :, full_model.sensor_indices]
            new_obs.append(new_obs_w.cpu().numpy())
            new_full.append(new_full_w.cpu().numpy())
    prep.hf_train = HFWindowDataset(np.vstack(new_obs), np.vstack(new_full))

    loader = DataLoader(prep.hf_valid, batch_size=64)
    new_obs, new_full = [], []
    with torch.no_grad():
        for obs_w, full_w in loader:
            obs_w = obs_w.to(device)
            total_corr, _ = full_model.forward_hf(obs_w, n_layers=n_active)
            new_full_w = full_w.to(device) - total_corr
            new_obs_w = new_full_w[:, :, full_model.sensor_indices]
            new_obs.append(new_obs_w.cpu().numpy())
            new_full.append(new_full_w.cpu().numpy())
    prep.hf_valid = HFWindowDataset(np.vstack(new_obs), np.vstack(new_full))


###############################################################################
# CLI ENTRY POINT
###############################################################################

if __name__ == "__main__":
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument("--scenario", default="iberian_event",
                        choices=["iberian_event", "italian_ambient",
                                 "italian_with_event"])
    parser.add_argument("--data-dir", default=None)
    parser.add_argument("--lf-mode", default="sindy_shred",  # make sure default="sindy_shred" it want to run sindy
                        choices=["sindy_shred", "vanilla_shred"],
                        help="LF pathway: sindy_shred (GRU+SINDy) or "
                             "vanilla_shred (LSTM only)")
    parser.add_argument("--lf-epochs", type=int, default=500)
    parser.add_argument("--hf-epochs", type=int, default=400)
    parser.add_argument("--max-samples", type=int, default=20000, # None
                        help="Truncate data to first N samples (e.g. 600)")
    args = parser.parse_args()

    cfg = PipelineConfig(
        scenario=args.scenario,
        data_dir=args.data_dir or f"data/synthetic/{args.scenario}",
        lf_mode=args.lf_mode,
        lf_epochs=args.lf_epochs,
        hf_epochs=args.hf_epochs,
        max_samples=args.max_samples,
    )

    # Scenario-specific adjustments
    if "italian" in args.scenario:
        cfg.n_pmu_total = 19
        cfg.t_hf = 100
        cfg.max_freq_hz = 12.0
        cfg.lags = 100
        cfg.lf_cutoff_hz = 12.0   # below 0.3 Hz North-South mode
    else:  # iberian
        cfg.n_pmu_total = 22
        cfg.t_hf = 100
        cfg.max_freq_hz = 5.0
        cfg.lags = 52
        cfg.lf_cutoff_hz = 5.0 # 5.0=max_freq_hz  # below 0.15 Hz East-West mode

    run_pipeline(cfg)